In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Lasso, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

df = pd.read_csv("top2019.csv")

X = df[["valence", "tempo", "energy"]]
y = df["danceability"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

param_grid = {"alpha": [0.001, 0.01, 0.1, 1, 10, 100]}
grid_lasso = GridSearchCV(Lasso(), param_grid, cv=5, scoring="neg_mean_squared_error")
grid_lasso.fit(X_train, y_train)

print("Best Lasso parameters:", grid_lasso.best_params_)
best_lasso = grid_lasso.best_estimator_

y_pred_lasso = best_lasso.predict(X_test)
print("R2", r2_score(y_test, y_pred_lasso))
print("RMSE", np.sqrt(mean_squared_error(y_test, y_pred_lasso)))
print("Features killed by Lasso", np.sum(best_lasso.coef_ == 0))

Best Lasso parameters: {'alpha': 0.001}
R2 -0.5954928144031519
RMSE 0.1445725117286533
Features killed by Lasso 0


In [6]:
param_grid = {"alpha": [0.001, 0.01, 0.1, 1, 10, 100]}
grid_ridge = GridSearchCV(Ridge(), param_grid, cv=5, scoring="neg_mean_squared_error")
grid_ridge.fit(X_train, y_train)

print("Best Ridge parameters", grid_ridge.best_params_)
best_ridge = grid_ridge.best_estimator_

y_pred_ridge = best_ridge.predict(X_test)
print("R2", r2_score(y_test, y_pred_ridge))
print("RMSE", np.sqrt(mean_squared_error(y_test, y_pred_ridge)))
print("Coefficients", best_ridge.coef_)

Best Ridge parameters {'alpha': 10}
R2 -0.5285724866881527
RMSE 0.1415081062950553
Coefficients [0.04081714 0.01582995 0.02766053]


The coefficients are all very small (0.04, 0.016, 0.028), meaning intense shrinkage. However, the R² of -0.53 is still negative, meaning the model still performs worse than just guessing the mean. The RMSE of 0.14 is similar to Lasso. Neither model is good at predicting danceability from these three features.